# Handwritten Digit Recognition with MNIST
### Fully Connected Neural Network using TensorFlow/Keras

This notebook covers:
1. Loading and preprocessing the MNIST dataset
2. Building a fully connected neural network
3. Training the model
4. Evaluating performance with confusion matrix
5. Hyperparameter tuning experiment

## Step 1 — Install & Import Libraries

In [ ]:
# Install required packages (uncomment if needed in Colab)
# !pip install tensorflow matplotlib seaborn scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import confusion_matrix, classification_report

print(f'TensorFlow version: {tf.__version__}')
print(f'Keras version: {keras.__version__}')

## Step 2 — Load and Preprocess the MNIST Dataset

In [ ]:
# Load MNIST dataset (auto-downloaded by Keras)
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print('=== Raw Dataset Shapes ===')
print(f'Training images : {X_train.shape}  ->  {X_train.shape[0]} images of {X_train.shape[1]}x{X_train.shape[2]} pixels')
print(f'Training labels : {y_train.shape}')
print(f'Test images     : {X_test.shape}')
print(f'Test labels     : {y_test.shape}')
print(f'Pixel range before normalization: [{X_train.min()}, {X_train.max()}]')

In [ ]:
# --- Normalize pixel values to [0, 1] ---
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

# --- One-hot encode labels ---
NUM_CLASSES = 10
y_train_ohe = to_categorical(y_train, NUM_CLASSES)
y_test_ohe  = to_categorical(y_test,  NUM_CLASSES)

print('=== After Preprocessing ===')
print(f'Pixel range after normalization : [{X_train.min():.1f}, {X_train.max():.1f}]')
print(f'Label shape (one-hot)           : {y_train_ohe.shape}')
print(f'Example label (digit {y_train[0]})       : {y_train_ohe[0]}')

In [ ]:
# --- Display 25 sample images with labels ---
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
fig.suptitle('MNIST Sample Images', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Label: {y_train[i]}', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# --- Show class distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, labels, title in zip(axes, [y_train, y_test], ['Training Set', 'Test Set']):
    counts = np.bincount(labels)
    bars = ax.bar(range(10), counts, color=plt.cm.tab10(np.linspace(0, 1, 10)), edgecolor='black')
    ax.set_title(f'Class Distribution — {title}', fontweight='bold')
    ax.set_xlabel('Digit')
    ax.set_ylabel('Count')
    ax.set_xticks(range(10))
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                str(count), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Step 3 — Build the Fully Connected Neural Network

In [ ]:
def build_model(hidden1=128, hidden2=64, dropout_rate=0.2, learning_rate=0.001):
    """
    Build a fully connected (Dense) neural network for digit classification.
    
    Architecture:
        Input   : 28x28 image  (784 values after flattening)
        Hidden1 : Dense(hidden1, ReLU) + Dropout
        Hidden2 : Dense(hidden2, ReLU) + Dropout
        Output  : Dense(10, Softmax)   -> probability per digit class
    """
    model = keras.Sequential([
        # --- Flatten 28x28 -> 784 ---
        layers.Flatten(input_shape=(28, 28), name='flatten'),

        # --- Hidden Layer 1 ---
        layers.Dense(hidden1, activation='relu', name='hidden_1'),
        layers.Dropout(dropout_rate, name='dropout_1'),

        # --- Hidden Layer 2 ---
        layers.Dense(hidden2, activation='relu', name='hidden_2'),
        layers.Dropout(dropout_rate, name='dropout_2'),

        # --- Output Layer (10 classes) ---
        layers.Dense(NUM_CLASSES, activation='softmax', name='output')
    ], name='MNIST_FC_Network')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


# Build the default model
model = build_model()
model.summary()

## Step 4 — Train the Neural Network

In [ ]:
EPOCHS     = 10
BATCH_SIZE = 128
VAL_SPLIT  = 0.1   # 10% of training data used for validation

print(f'Training for {EPOCHS} epochs | Batch size: {BATCH_SIZE} | Validation split: {VAL_SPLIT}')
print('-' * 60)

history = model.fit(
    X_train, y_train_ohe,
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    validation_split= VAL_SPLIT,
    verbose         = 1
)

In [ ]:
# --- Plot training curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS + 1)

# Accuracy
ax1.plot(epochs_range, history.history['accuracy'],     'b-o', label='Train Accuracy',      linewidth=2)
ax1.plot(epochs_range, history.history['val_accuracy'], 'r-o', label='Validation Accuracy', linewidth=2)
ax1.set_title('Model Accuracy over Epochs', fontweight='bold', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0.9, 1.0])

# Loss
ax2.plot(epochs_range, history.history['loss'],     'b-o', label='Train Loss',      linewidth=2)
ax2.plot(epochs_range, history.history['val_loss'], 'r-o', label='Validation Loss', linewidth=2)
ax2.set_title('Model Loss over Epochs', fontweight='bold', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Training History', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 5 — Evaluate the Model's Performance

In [ ]:
# --- Test set evaluation ---
test_loss, test_acc = model.evaluate(X_test, y_test_ohe, verbose=0)
print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')

In [ ]:
# --- Predictions ---
y_pred_probs = model.predict(X_test, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)   # predicted digit
y_true       = y_test                             # true digit

# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=range(10), yticklabels=range(10),
    linewidths=0.5, ax=ax
)
ax.set_title('Confusion Matrix — Test Set', fontsize=15, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- Per-class metrics ---
print('Classification Report:\n')
print(classification_report(y_true, y_pred, target_names=[f'Digit {i}' for i in range(10)]))

# --- Identify hardest digits ---
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
print('\nPer-Digit Accuracy:')
for digit, acc in enumerate(per_class_accuracy):
    bar = '#' * int(acc * 30)
    print(f'  Digit {digit}: {acc*100:5.2f}%  |{bar}')

hardest = np.argmin(per_class_accuracy)
print(f'\nHardest digit to classify: {hardest} ({per_class_accuracy[hardest]*100:.2f}% accuracy)')

In [ ]:
# --- Visualize misclassified examples ---
wrong_idx = np.where(y_pred != y_true)[0]
print(f'Total misclassified: {len(wrong_idx)} / {len(y_true)}  ({len(wrong_idx)/len(y_true)*100:.2f}%)')

# Show first 20 mistakes
n_show = min(20, len(wrong_idx))
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
fig.suptitle('Misclassified Examples', fontsize=15, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i >= n_show:
        ax.axis('off')
        continue
    idx = wrong_idx[i]
    confidence = y_pred_probs[idx, y_pred[idx]] * 100
    ax.imshow(X_test[idx], cmap='gray')
    ax.set_title(f'True: {y_true[idx]}  Pred: {y_pred[idx]}\nConf: {confidence:.1f}%',
                 fontsize=9, color='red')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 6 — Hyperparameter Tuning Experiment

We compare 4 configurations varying hidden layer sizes and learning rate.

In [ ]:
# Define configurations to test
configs = [
    {'hidden1': 64,  'hidden2': 32,  'learning_rate': 0.001, 'label': 'Small  LR=0.001'},
    {'hidden1': 128, 'hidden2': 64,  'learning_rate': 0.001, 'label': 'Medium LR=0.001'},
    {'hidden1': 256, 'hidden2': 128, 'learning_rate': 0.001, 'label': 'Large  LR=0.001'},
    {'hidden1': 128, 'hidden2': 64,  'learning_rate': 0.01,  'label': 'Medium LR=0.01 '},
]

results = []

for cfg in configs:
    print(f"\nTraining: {cfg['label']}")
    print('-' * 40)
    m = build_model(
        hidden1       = cfg['hidden1'],
        hidden2       = cfg['hidden2'],
        learning_rate = cfg['learning_rate']
    )
    hist = m.fit(
        X_train, y_train_ohe,
        epochs           = EPOCHS,
        batch_size       = BATCH_SIZE,
        validation_split = VAL_SPLIT,
        verbose          = 0   # quiet training for comparison runs
    )
    _, test_accuracy = m.evaluate(X_test, y_test_ohe, verbose=0)
    results.append({
        'label'         : cfg['label'],
        'history'       : hist,
        'test_accuracy' : test_accuracy
    })
    print(f"  Test accuracy: {test_accuracy*100:.2f}%")

print('\nDone!')

In [ ]:
# --- Plot hyperparameter comparison ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for res, color in zip(results, colors):
    val_acc = res['history'].history['val_accuracy']
    val_loss = res['history'].history['val_loss']
    ax1.plot(range(1, EPOCHS+1), val_acc,  '-o', label=res['label'], color=color, linewidth=2, markersize=5)
    ax2.plot(range(1, EPOCHS+1), val_loss, '-o', label=res['label'], color=color, linewidth=2, markersize=5)

ax1.set_title('Validation Accuracy — Hyperparameter Comparison', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Validation Accuracy')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.set_title('Validation Loss — Hyperparameter Comparison', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation Loss')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle('Hyperparameter Tuning Results', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table
print('\n=== Final Test Accuracy Summary ===')
print(f'{"Configuration":<25} {"Test Accuracy":>15}')
print('-' * 42)
best = max(results, key=lambda r: r['test_accuracy'])
for res in sorted(results, key=lambda r: r['test_accuracy'], reverse=True):
    marker = ' <-- BEST' if res['label'] == best['label'] else ''
    print(f"{res['label']:<25} {res['test_accuracy']*100:>14.2f}%{marker}")

## Step 7 — Predict on Individual Images

In [ ]:
# Pick a few test images and show detailed predictions
sample_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
fig.suptitle('Per-Class Probability Bars for Sample Predictions', fontsize=14, fontweight='bold')

for ax, idx in zip(axes.flat, sample_indices):
    probs      = y_pred_probs[idx]
    pred_digit = np.argmax(probs)
    true_digit = y_true[idx]
    color      = 'green' if pred_digit == true_digit else 'red'

    bar_colors = ['#1f77b4'] * 10
    bar_colors[pred_digit] = color

    ax.bar(range(10), probs, color=bar_colors, edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(10))
    ax.set_ylim([0, 1])
    ax.set_title(f'True: {true_digit}  |  Pred: {pred_digit}', fontweight='bold',
                 color=color, fontsize=10)
    ax.set_xlabel('Digit Class')
    ax.set_ylabel('Probability')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

| Step | What we did |
|------|-------------|
| 1 | Loaded MNIST — 60,000 train / 10,000 test images |
| 2 | Normalized pixels to [0,1] and one-hot encoded labels |
| 3 | Built FC network: Flatten → Dense(128, ReLU) → Dense(64, ReLU) → Dense(10, Softmax) |
| 4 | Trained for 10 epochs with Adam optimizer & categorical cross-entropy loss |
| 5 | Evaluated with confusion matrix — identified hardest digits |
| 6 | Ran hyperparameter tuning across 4 configurations |

Typical result: **~98% test accuracy** on the default model.

### Key Observations
- Digits **4, 7, 9** are often confused with each other
- Larger hidden layers improve accuracy but risk overfitting
- Higher learning rate (0.01) trains faster but may be less stable
- Dropout regularization prevents overfitting on training data